# food-cap-standard: pre-merge test (2026-10-23)

Branch `food-cap-standard`. N's decision: the capped mouthful (`food_cap=1`) is the standard model; the legacy rule (`food_cap=0`) is not part of the model. Checks, each entering through a public door:

1. Every loader gives the standard rule by default: `EvoCA.init`, a recipe without a `food_cap` key, a pre-A9 scan row through `params_from_row`, `evoca_from_scan`, `sim_from_scan` and a gallery `Entry`; an explicit legacy `food_cap=0` is honored; a notice is printed where the row's metrics came from the legacy rule.
2. The rescan `Scans/2026-10-23_rescan_0427_standard` covers exactly the 600 rows of the three 04-27 scans with their parameters, standard rule, no errors.
3. Determinism: its `food_cap=1` push_edges m_scale 1.2 rows equal the independent 65-row rescan, and re-running one row now reproduces its metrics to the precision the CSV keeps.
4. The Reaction-diffusion notebook: executed without errors, and no code above the legacy section sets `food_cap=0`.

In [1]:
import sys, os, io, json, contextlib, tempfile
from pathlib import Path
import numpy as np, pandas as pd
p = Path.cwd().resolve()
while not (p / 'C' / 'evoca.h').exists(): p = p.parent
ROOT = p; sys.path.insert(0, str(ROOT / 'python'))
import evoca_explore as ee, evoca_nb as nb
from evoca_py import EvoCA, import_run
from evoca_gallery import Entry, Gallery
RD = ROOT / 'Research' / 'Reaction-diffusion'; SC = RD / 'Scans'
def check(name, ok): print(('PASS' if ok else 'FAIL') + ': ' + name); assert ok

## 1. Default rule at every door

In [2]:
s = EvoCA(); s.init(16); check('EvoCA.init default food_cap == 1', s.get_food_cap() == 1); s.free()

out = io.StringIO()
with contextlib.redirect_stdout(out):
    sim, _ = import_run(str(RD / 'Runs' / '2026-04-27_top_2_cfg246_score0.88.evoca'), N=16)   # no food_cap key, m_scale 1.2
check('recipe without food_cap key loads food_cap=1', sim.get_food_cap() == 1); sim.free()
check('... with the standard-rule notice', 'running the standard capped rule' in out.getvalue())

d = pd.read_csv(SC / '2026-04-27_push_edges' / 'results.csv').set_index('config_idx')
assert 'param_food_cap' not in d.columns and d.loc[246, 'param_m_scale'] == 1.2
out = io.StringIO()
with contextlib.redirect_stdout(out):
    pr = nb.params_from_row(d.loc[246])
check('params_from_row: pre-A9 row -> food_cap=1', pr['food_cap'] == 1)
check('... notice says the metrics are legacy-rule', 'measured under the legacy rule' in out.getvalue())
with contextlib.redirect_stdout(io.StringIO()):
    pl = nb.params_from_row(d[d.param_m_scale <= 1].iloc[0])
check('params_from_row: m_scale <= 1 row also food_cap=1', pl['food_cap'] == 1)

with tempfile.TemporaryDirectory() as t, contextlib.redirect_stdout(io.StringIO()):
    rec = json.load(open(ee.evoca_from_scan(SC / '2026-04-27_push_edges', 246, runs_dir=t)))
check('evoca_from_scan writes food_cap=1 explicitly', rec['metaparams_init']['food_cap'] == 1)

with contextlib.redirect_stdout(io.StringIO()):
    sim, _ = nb.sim_from_scan(SC / '2026-04-27_push_edges', 246, N=16)
check('sim_from_scan runs the standard rule', sim.get_food_cap() == 1); sim.free()
sim, _ = Entry(scan='2026-04-27_push_edges', cfg=246).load(N=16, quiet=True)
check('gallery Entry(scan=...) runs the standard rule', sim.get_food_cap() == 1); sim.free()
sim, _ = Entry(scan='2026-04-27_push_edges', cfg=246, overrides={'food_cap': 0}).load(N=16)
check('explicit legacy override honored', sim.get_food_cap() == 0); sim.free()
sim, _ = Entry(recipe='2026-10-21_rd_waves_patchy_cfg246.evoca').load(N=16)
check('a recipe that records food_cap=0 keeps it (legacy section only)', sim.get_food_cap() == 0); sim.free()
t = Gallery('t', [Entry(recipe='2026-04-27_top_2_cfg246_score0.88.evoca'), Entry(scan='2026-04-27_push_edges', cfg=246, label='row')]).table()
check('gallery table: both doors agree (RXD-D4)', list(t.food_cap) == [1, 1])

PASS: EvoCA.init default food_cap == 1
PASS: recipe without food_cap key loads food_cap=1
PASS: ... with the standard-rule notice
PASS: params_from_row: pre-A9 row -> food_cap=1
PASS: ... notice says the metrics are legacy-rule
PASS: params_from_row: m_scale <= 1 row also food_cap=1
PASS: evoca_from_scan writes food_cap=1 explicitly
PASS: sim_from_scan runs the standard rule
PASS: gallery Entry(scan=...) runs the standard rule
PASS: explicit legacy override honored
PASS: a recipe that records food_cap=0 keeps it (legacy section only)
PASS: gallery table: both doors agree (RXD-D4)


## 2. The rescan covers the 04-27 rows

In [3]:
R = pd.read_csv(SC / '2026-10-23_rescan_0427_standard' / 'results.csv')
src = pd.concat([pd.read_csv(SC / f'2026-04-27_{s}' / 'results.csv').assign(scan=s) for s in ('initial', 'refined', 'push_edges')])
check('600 rows, no errors', len(R) == 600 and R.error.isna().all())
check('all rows food_cap=1', (R.param_food_cap == 1).all())
check('same (scan, config_idx) set as the 04-27 scans', set(zip(R.scan, R.config_idx)) == set(zip(src.scan, src.config_idx)))
m = R.merge(src, on=['scan', 'config_idx'], suffixes=('', '_src'))
same = all(np.allclose(m[f'param_{k}'], m[f'param_{k}_src']) for k in ('food_inc', 'm_scale', 'gdiff', 'tax', 'mu_lut'))
check('parameters match the source rows (food_inc, m_scale, gdiff, tax, mu_lut)', same)
check('mu_egenome column read as mu_egene', np.allclose(m.param_mu_egene, m.param_mu_egenome))

PASS: 600 rows, no errors
PASS: all rows food_cap=1
PASS: same (scan, config_idx) set as the 04-27 scans
PASS: parameters match the source rows (food_inc, m_scale, gdiff, tax, mu_lut)
PASS: mu_egenome column read as mu_egene


## 3. Determinism

In [4]:
M = ['alive_density_mean', 'correlation_length_mean', 'largest_patch_temporal_std', 'n_distinct_genomes_temporal_std', 'unique_top_genomes']
a = pd.read_csv(SC / '2026-10-23_push_edges_m1p2_rescan' / 'results.csv').query('param_food_cap == 1').set_index('config_idx')
b = R.query("scan == 'push_edges'").set_index('config_idx').loc[a.index]
check('independent rescans agree on the 65 m_scale 1.2 rows', all(np.array_equal(a[k].values, b[k].values) for k in M))
row = R.query("scan == 'refined'").iloc[7]
with contextlib.redirect_stdout(io.StringIO()):
    p = nb.params_from_row(src.query("scan == 'refined'").set_index('config_idx').loc[row.config_idx])
r = ee.run_sim(p, n_steps=5000, sample_every=100, N=256, seed=0, shadow=True, init='halfplane')
check(f'refined cfg{row.config_idx} re-run now reproduces the rescan row (to CSV precision)', all(np.isclose(float(r[k]), float(row[k]), rtol=1e-12, atol=0) for k in M))   # CSV text keeps ~16 digits

PASS: independent rescans agree on the 65 m_scale 1.2 rows


PASS: refined cfg7 re-run now reproduces the rescan row (to CSV precision)


## 4. The notebook

In [5]:
import nbformat
book = nbformat.read(RD / 'Reaction-diffusion.ipynb', 4)
errs = [i for i, c in enumerate(book.cells) if c.cell_type == 'code' for o in c.outputs if o.output_type == 'error']
check('notebook executed without errors', not errs)
leg = [i for i, c in enumerate(book.cells) if c.source.startswith('## Legacy rule')]
check('one legacy section', len(leg) == 1)
import ast
def sets_legacy(src):
    """True if code (not strings or comments) passes food_cap=0 as a keyword or a dict entry."""
    src = '\n'.join(l for l in src.split('\n') if not l.lstrip().startswith(('%', '!')))
    for n in ast.walk(ast.parse(src)):
        if isinstance(n, ast.keyword) and n.arg == 'food_cap' and isinstance(n.value, ast.Constant) and n.value.value == 0:
            return True
        if isinstance(n, ast.Dict) and any(isinstance(k, ast.Constant) and k.value == 'food_cap' and isinstance(v, ast.Constant) and v.value == 0
                                           for k, v in zip(n.keys, n.values)):
            return True
    return False
check('the check detects a legacy setting', sets_legacy("Entry(x, overrides={'food_cap': 0})") and sets_legacy("make_sim(8, food_cap=0)") and not sets_legacy('"food_cap=0"'))
main = [c.source for c in book.cells[:leg[0]] if c.cell_type == 'code']
check('no main-thread code sets food_cap=0', not any(sets_legacy(s) for s in main))
check('the legacy section does set it', any(sets_legacy(c.source) for c in book.cells[leg[0]:] if c.cell_type == 'code'))
check('every code cell executed', all(c.execution_count for c in book.cells if c.cell_type == 'code' and c.source.strip()))
print('ALL PASS')

PASS: notebook executed without errors
PASS: one legacy section
PASS: the check detects a legacy setting
PASS: no main-thread code sets food_cap=0
PASS: the legacy section does set it
PASS: every code cell executed
ALL PASS
